In [42]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/X_test_Kaggle.csv
/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/X_train_Kaggle.csv
/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/y_train_Kaggle.csv
/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/sample_submission_credit.csv


In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
#modelo de clasificacion
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
#modelo de regresion
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor 

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [44]:


X_train = pd.read_csv(
    "/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/X_train_Kaggle.csv"
    ,low_memory=False
)

y_train = pd.read_csv(
    "/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/y_train_Kaggle.csv"
)

X_test = pd.read_csv(
    "/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/X_test_Kaggle.csv"
)

sample_submission = pd.read_csv(
    "/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/sample_submission_credit.csv"
)

In [45]:
y = y_train["Credit_Score"]

In [46]:
X_train["Monthly_Balance"] = pd.to_numeric(
    X_train["Monthly_Balance"],
    errors="coerce"
)

X_test["Monthly_Balance"] = pd.to_numeric(
    X_test["Monthly_Balance"],
    errors="coerce"
)
print(X_train["Monthly_Balance"].dtype)

print(
    X_train["Monthly_Balance"].isnull().sum()
)

print(
    X_test["Monthly_Balance"].isnull().sum()
)

float64
840
369


In [47]:
numeric_to_convert = [
    "Age",
    "Annual_Income",
    "Num_of_Loan",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Outstanding_Debt",
    "Amount_invested_monthly"
]

for col in numeric_to_convert:
    X_train[col] = (
        X_train[col]
        .astype("string")
        .str.strip()
        .str.replace("_", "", regex=False)
    )

    X_test[col] = (
        X_test[col]
        .astype("string")
        .str.strip()
        .str.replace("_", "", regex=False)
    )
    X_train[col] = pd.to_numeric(
        X_train[col],
        errors="coerce"
    )

    X_test[col] = pd.to_numeric(
        X_test[col],
        errors="coerce"
    )

print(X_train[numeric_to_convert].dtypes)


Age                          Int64
Annual_Income              Float64
Num_of_Loan                  Int64
Num_of_Delayed_Payment       Int64
Changed_Credit_Limit       Float64
Outstanding_Debt           Float64
Amount_invested_monthly    Float64
dtype: object


In [48]:
columns_to_drop = [
    "ID",
    "Customer_ID",
    "Name",
    "SSN"
]

X_train = X_train.drop(
    columns=columns_to_drop,
    errors="ignore"
)

X_test = X_test.drop(
    columns=columns_to_drop,
    errors="ignore"
)

In [49]:
numeric_cols = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    exclude="number"
).columns.tolist()

print("NUMÉRICAS:")
print(numeric_cols)

print("\nCATEGÓRICAS:")
print(categorical_cols)

NUMÉRICAS:
['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']

CATEGÓRICAS:
['Month', 'Occupation', 'Type_of_Loan', 'Credit_Mix', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Payment_Behaviour']


In [50]:
#Divide de models 
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
numeric_cols = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    exclude="number"
).columns.tolist()

#preprocessor
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median"
    )),
    ("scaler", StandardScaler())
])

#categorical values
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

#combine
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print(" correctly")


 correctly


In [51]:
from sklearn.dummy import DummyClassifier
models = {
    "Baseline": DummyClassifier(
        strategy="most_frequent"
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
}
# train
results = []
trained_models = {}

for name, estimator in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipeline.fit(X_fit, y_fit)

    pred = pipeline.predict(X_val)

    accuracy = accuracy_score(y_val, pred)
    macro_f1 = f1_score(
        y_val, pred,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Macro F1": macro_f1
    })

    trained_models[name] = pipeline


results_df = pd.DataFrame(results)

display(results_df.sort_values(
    by="Macro F1",
    ascending=False
))

best_name = results_df.sort_values(
    "Macro F1", ascending=False
).iloc[0]["Model"]

best_model = trained_models[best_name]

print("Best model:", best_name)

print(classification_report(
    y_val,
    best_model.predict(X_val),
    zero_division=0
))


,Model,Accuracy,Macro F1
2,Random Forest,0.675000,0.646008
1,Logistic Regression,0.629000,0.591879
0,Baseline,0.532857,0.231749


Best model: Random Forest
              precision    recall  f1-score   support

        Good       0.59      0.51      0.55      2501
        Poor       0.67      0.68      0.67      4039
    Standard       0.70      0.73      0.71      7460

    accuracy                           0.68     14000
   macro avg       0.65      0.64      0.65     14000
weighted avg       0.67      0.68      0.67     14000



In [58]:

# ==========================================
# PART 5 - FINAL KAGGLE SUBMISSION
# ==========================================

import pandas as pd
from sklearn.base import clone

# 1. REENTRENAR EL MEJOR MODELO
sample = pd.read_csv(
    "/kaggle/input/competitions/iisem-2026-artificial-intelligence-midterm-exam/sample_submission_credit.csv"
)
final_model = clone(best_model)

final_model.fit(X_train, y)

# 2. PREDECIR TEST

test_predictions = final_model.predict(X_test)

print("\nPredictions:", len(test_predictions))

print(pd.Series(test_predictions).value_counts())


# 3. CREAR SUBMISSION

submission = sample.copy()

# OPCIÓN A:
# Si Kaggle acepta Good, Poor y Standard

submission["class"] = test_predictions
assert len(submission) == len(X_test)

assert submission.columns.tolist() == [
    "id", "class"
]

assert not submission["class"].isnull().any()

print("\nVista previa:")
display(submission.head())

print("\nDimensiones:", submission.shape)

print("\nDistribución de predicciones:")
print(submission["class"].value_counts())


submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("\nArchivo creado correctamente:")
print("/kaggle/working/submission.csv")



Predictions: 30000
Standard    15691
Poor         9110
Good         5199
Name: count, dtype: int64

Vista previa:


,id,class
0,1,Good
1,2,Poor
2,3,Good
3,4,Poor
4,5,Good



Dimensiones: (30000, 2)

Distribución de predicciones:
class
Standard    15691
Poor         9110
Good         5199
Name: count, dtype: int64

Archivo creado correctamente:
/kaggle/working/submission.csv
